# MTG Cards DB Builder (Scryfall)
Pull Scryfall **Bulk Data** and load a deck-building-focused subset of fields into a local **SQLite** database.

**Tables**
- `oracle_cards` (one row per `oracle_id`)
- `printings` (one row per Scryfall `id`)
- `card_faces` (multi-faced)


In [1]:
# If running on a fresh environment, uncomment:
# !pip install requests tqdm

import json
import sqlite3
import time
from pathlib import Path

import requests
from tqdm.auto import tqdm


In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "mtg_cards.sqlite3"

# For testing, set to a small number like 5000 to scan only the first N rows
# from the bulk file (before legality filtering). Set to None for a full load.
MAX_CARDS_TO_SCAN = None

## Discover bulk download URL

In [6]:
BULK_ENDPOINT = "https://api.scryfall.com/bulk-data"
bulk = requests.get(BULK_ENDPOINT, timeout=60).json()
[b["type"] for b in bulk["data"]]


['oracle_cards', 'unique_artwork', 'default_cards', 'all_cards', 'rulings']

In [8]:
BULK_TYPE = "default_cards"  # or "oracle_cards"

target = next(b for b in bulk["data"] if b["type"] == BULK_TYPE)
download_uri = target["download_uri"]
updated_at = target.get("updated_at")
size_bytes = target.get("size")

download_uri, updated_at, size_bytes


('https://data.scryfall.io/default-cards/default-cards-20260306100900.json',
 '2026-03-06T10:09:00.087+00:00',
 528524239)

## Download the bulk JSON

In [11]:
out_path = DATA_DIR / f"{BULK_TYPE}.json"

def download_file(url: str, dest: Path, chunk_size: int = 1 << 20):
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("Content-Length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=f"Downloading {dest.name}") as pbar:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

if not out_path.exists():
    download_file(download_uri, out_path)
else:
    print(f"Already exists: {out_path} (delete it to re-download)")


## Create the database schema

In [14]:
SCHEMA_SQL = """
PRAGMA journal_mode=WAL;

CREATE TABLE IF NOT EXISTS oracle_cards (
  oracle_id TEXT PRIMARY KEY,
  name TEXT,
  mana_cost TEXT,
  cmc REAL,
  type_line TEXT,
  oracle_text TEXT,
  power TEXT,
  toughness TEXT,
  loyalty TEXT,
  colors TEXT,           -- JSON array
  color_identity TEXT,   -- JSON array
  color_indicator TEXT,  -- JSON array or null
  keywords TEXT,         -- JSON array (may be missing in some data)
  produced_mana TEXT,    -- JSON array (optional)
  reserved INTEGER,
  edhrec_rank INTEGER,
  layout TEXT
);

CREATE TABLE IF NOT EXISTS printings (
  id TEXT PRIMARY KEY,
  oracle_id TEXT,
  name TEXT,
  set_code TEXT,
  set_name TEXT,
  collector_number TEXT,
  rarity TEXT,
  released_at TEXT,
  lang TEXT,
  games TEXT,            -- JSON array
  digital INTEGER,
  booster INTEGER,
  legalities TEXT,       -- JSON object
  image_uris TEXT,       -- JSON object (single-faced)
  scryfall_uri TEXT,
  FOREIGN KEY (oracle_id) REFERENCES oracle_cards(oracle_id)
);

CREATE TABLE IF NOT EXISTS card_faces (
  printing_id TEXT,
  face_index INTEGER,
  name TEXT,
  mana_cost TEXT,
  type_line TEXT,
  oracle_text TEXT,
  power TEXT,
  toughness TEXT,
  loyalty TEXT,
  colors TEXT,          -- JSON array
  color_indicator TEXT, -- JSON array
  image_uris TEXT,      -- JSON object
  PRIMARY KEY (printing_id, face_index),
  FOREIGN KEY (printing_id) REFERENCES printings(id)
);

CREATE INDEX IF NOT EXISTS idx_printings_oracle_id ON printings(oracle_id);
CREATE INDEX IF NOT EXISTS idx_oracle_name ON oracle_cards(name);
CREATE INDEX IF NOT EXISTS idx_printings_set ON printings(set_code);
"""


In [16]:
conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA_SQL)
conn.commit()
print("DB ready at:", DB_PATH.resolve())


DB ready at: /Users/heathverhasselt/data/mtg_cards.sqlite3


## Load cards into SQLite

In [19]:
def jdump(x):
    return None if x is None else json.dumps(x, ensure_ascii=False)

def upsert_oracle(cur, c):
    cur.execute(
        '''
        INSERT INTO oracle_cards (
          oracle_id, name, mana_cost, cmc, type_line, oracle_text,
          power, toughness, loyalty,
          colors, color_identity, color_indicator,
          keywords, produced_mana, reserved, edhrec_rank, layout
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        ON CONFLICT(oracle_id) DO UPDATE SET
          name=COALESCE(excluded.name, oracle_cards.name),
          mana_cost=COALESCE(excluded.mana_cost, oracle_cards.mana_cost),
          cmc=COALESCE(excluded.cmc, oracle_cards.cmc),
          type_line=COALESCE(excluded.type_line, oracle_cards.type_line),
          oracle_text=COALESCE(excluded.oracle_text, oracle_cards.oracle_text),
          power=COALESCE(excluded.power, oracle_cards.power),
          toughness=COALESCE(excluded.toughness, oracle_cards.toughness),
          loyalty=COALESCE(excluded.loyalty, oracle_cards.loyalty),
          colors=COALESCE(excluded.colors, oracle_cards.colors),
          color_identity=COALESCE(excluded.color_identity, oracle_cards.color_identity),
          color_indicator=COALESCE(excluded.color_indicator, oracle_cards.color_indicator),
          keywords=COALESCE(excluded.keywords, oracle_cards.keywords),
          produced_mana=COALESCE(excluded.produced_mana, oracle_cards.produced_mana),
          reserved=COALESCE(excluded.reserved, oracle_cards.reserved),
          edhrec_rank=COALESCE(excluded.edhrec_rank, oracle_cards.edhrec_rank),
          layout=COALESCE(excluded.layout, oracle_cards.layout)
        ''',
        (
            c.get("oracle_id"),
            c.get("name"),
            c.get("mana_cost"),
            c.get("cmc"),
            c.get("type_line"),
            c.get("oracle_text"),
            c.get("power"),
            c.get("toughness"),
            c.get("loyalty"),
            jdump(c.get("colors")),
            jdump(c.get("color_identity")),
            jdump(c.get("color_indicator")),
            jdump(c.get("keywords")),
            jdump(c.get("produced_mana")),
            1 if c.get("reserved") else 0 if c.get("reserved") is not None else None,
            c.get("edhrec_rank"),
            c.get("layout"),
        ),
    )

def upsert_printing(cur, c):
    cur.execute(
        '''
        INSERT INTO printings (
          id, oracle_id, name,
          set_code, set_name, collector_number, rarity,
          released_at, lang,
          games, digital, booster,
          legalities, image_uris, scryfall_uri
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        ON CONFLICT(id) DO UPDATE SET
          oracle_id=excluded.oracle_id,
          name=excluded.name,
          set_code=excluded.set_code,
          set_name=excluded.set_name,
          collector_number=excluded.collector_number,
          rarity=excluded.rarity,
          released_at=excluded.released_at,
          lang=excluded.lang,
          games=excluded.games,
          digital=excluded.digital,
          booster=excluded.booster,
          legalities=excluded.legalities,
          image_uris=excluded.image_uris,
          scryfall_uri=excluded.scryfall_uri
        ''',
        (
            c.get("id"),
            c.get("oracle_id"),
            c.get("name"),
            c.get("set"),
            c.get("set_name"),
            c.get("collector_number"),
            c.get("rarity"),
            c.get("released_at"),
            c.get("lang"),
            jdump(c.get("games")),
            1 if c.get("digital") else 0 if c.get("digital") is not None else None,
            1 if c.get("booster") else 0 if c.get("booster") is not None else None,
            jdump(c.get("legalities")),
            jdump(c.get("image_uris")),
            c.get("scryfall_uri"),
        ),
    )

def upsert_faces(cur, c):
    faces = c.get("card_faces")
    if not faces:
        return
    for i, f in enumerate(faces):
        cur.execute(
            '''
            INSERT INTO card_faces (
              printing_id, face_index,
              name, mana_cost, type_line, oracle_text,
              power, toughness, loyalty,
              colors, color_indicator, image_uris
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)
            ON CONFLICT(printing_id, face_index) DO UPDATE SET
              name=excluded.name,
              mana_cost=excluded.mana_cost,
              type_line=excluded.type_line,
              oracle_text=excluded.oracle_text,
              power=excluded.power,
              toughness=excluded.toughness,
              loyalty=excluded.loyalty,
              colors=excluded.colors,
              color_indicator=excluded.color_indicator,
              image_uris=excluded.image_uris
            ''',
            (
                c.get("id"),
                i,
                f.get("name"),
                f.get("mana_cost"),
                f.get("type_line"),
                f.get("oracle_text"),
                f.get("power"),
                f.get("toughness"),
                f.get("loyalty"),
                jdump(f.get("colors")),
                jdump(f.get("color_indicator")),
                jdump(f.get("image_uris")),
            ),
        )


In [ ]:
# NOTE: json.load reads the full file into memory.
# If you want streaming, use `ijson` and insert incrementally.

with open(out_path, "r", encoding="utf-8") as f:
    cards = json.load(f)

scan_limit = MAX_CARDS_TO_SCAN if MAX_CARDS_TO_SCAN is not None else len(cards)
scan_limit = min(scan_limit, len(cards))

print(f"Loaded {len(cards):,} cards from bulk; scanning {scan_limit:,}...")

cur = conn.cursor()
t0 = time.time()
inserted = skipped = 0

for idx, c in enumerate(tqdm(cards[:scan_limit], desc="Inserting")):
    if not c.get("oracle_id") or not c.get("id"):
        skipped += 1
        continue
    # Only load English cards legal in Standard, Historic, Explorer, or Pioneer
    if c.get("lang") != "en":
        skipped += 1
        continue
    legalities = c.get("legalities", {})
    if not any(legalities.get(fmt) == "legal" for fmt in ("standard", "historic", "explorer", "pioneer")):
        skipped += 1
        continue

    upsert_oracle(cur, c)
    upsert_printing(cur, c)
    upsert_faces(cur, c)
    inserted += 1

    if inserted % 5000 == 0:
        conn.commit()

conn.commit()
print(f"Done in {round(time.time() - t0, 1)}s — inserted {inserted:,}, skipped {skipped:,}")

## Sanity checks

In [ ]:
cur = conn.cursor()
oracle_count = cur.execute("SELECT COUNT(*) FROM oracle_cards").fetchone()[0]
printing_count = cur.execute("SELECT COUNT(*) FROM printings").fetchone()[0]
faces_count = cur.execute("SELECT COUNT(*) FROM card_faces").fetchone()[0]

print(f"oracle_cards: {oracle_count:,}  (expect ~15,000+ across Standard, Historic, Explorer, Pioneer)")
print(f"printings:    {printing_count:,}")
print(f"card_faces:   {faces_count:,}")

In [24]:
# Example query: find cards with 'draw a card' in oracle text
q = "%draw a card%"
rows = conn.execute(
    '''
    SELECT o.name, o.type_line, o.mana_cost
    FROM oracle_cards o
    WHERE o.oracle_text LIKE ?
    LIMIT 20
    ''',
    (q,)
).fetchall()

rows[:10]


[('Surge of Brilliance', 'Instant', '{1}{U}'),
 ('Afflict', 'Instant', '{2}{B}'),
 ('The Convincing General',
  'Legendary Creature — Shapeshifter',
  '{W}{U}{B}{R}{G}'),
 ('Lonely Sandbar', 'Land', ''),
 ('Spark Spray', 'Instant', '{R}'),
 ('Remand', 'Instant', '{1}{U}'),
 ('Edric, Spymaster of Trest', 'Legendary Creature — Elf Rogue', '{1}{G}{U}'),
 ('Farid, Enterprising Salvager',
  'Legendary Creature — Human Soldier',
  '{2}{R}'),
 ('Briarbridge Patrol', 'Creature — Human Warrior', '{3}{G}'),
 ('Izoni, Thousand-Eyed',
  'Legendary Creature — Elf Shaman',
  '{2}{B}{B}{G}{G}')]